# <font color=firebrick>Econometrics — M2 MBFA</font>
## Session 2 / 8 (CM) — The OLS estimator, variance, and t-test

**Prerequisites:** Session 1 (why OLS, matrix notation, Monte Carlo simulation of a simple linear DGP).

**Today's objectives:**
- Derive the OLS estimator $\hat{\beta}$ analytically (least-squares criterion, first-order condition).
- Show that $\hat{\beta}$ is unbiased under the zero conditional mean assumption.
- Derive the variance of $\hat{\beta}$ and understand what drives its precision.
- Build and interpret a **t-test** to assess whether a coefficient is statistically significant.

Practice exercises for this session are provided in the companion TD notebook.

---

### Recap: recreating last session's simulated data

This notebook can be run on its own (fresh kernel). We first recreate the exact same simulated dataset (same seed, same $\beta$) that was generated in Session 1, so the rest of the session's code runs without depending on that earlier notebook.


In [ ]:
# Recap of Session 1: same random seed, same beta, same N as before
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=10)

beta = 1        # true coefficient of the DGP
N = 100         # number of observations

x = rng.exponential(1, size=N)
eps = rng.normal(0, 1, size=N)
y = x * beta + eps

print('We recreated the same simulated dataset as generated in Session 1. We now can use it in the following.')

We recreate the same simulated dataset as generated in Session 1. We now can use it in the following.


# 4. Estimator $\hat{\beta}$
Here we focus on the simple linear regression model with one explanatory variable:

$$
y_j = \beta x_{i} +\varepsilon_i, \quad i = 1, \dots, N.
$$

where:

- $y_i$ is the dependent variable (outcome) for observation $i$,
- $x_i$ is the explanatory variable for observation $i$,
- $\beta$ is the unknown parameter we want to estimate,
- $\varepsilon_i$ is the *true* (unobserved) error term that captures all other factors affecting $y_i$ othe than $x_i$. It represents the difference between the actual and predicted values for each observation.

Geometrically, we are looking for the straight line that “best fits” the cloud of points $(x_i,y_i)$:
$$
\hat{y_i}=\hat{\beta}x_i
$$

You can picture this by placing a ruler through the origin and rotating it until the line seems to follow the overall trend of the points. OLS formalizes this idea. Mathematically, the “best” line is the one that minimizes the total squared distance between the observed points and the predicted values - these vertical distances are called *residuals*.

### 4.1 The least-squares criterion
For each observation, define the residual **as a function of a candidate value $\beta$** - the vertical distance between the observed point $(x_i,y_i)$ and the candidate line $y=\beta x$:
$$
\hat{\varepsilon}_i(\beta) = y_i-\beta x_i, \qquad i=1,\dots,N.
$$

We now state the **least-squares program**. It is a minimization problem in $\beta$, whose solution $\hat\beta$ is the OLS estimator:
$$
\boxed{\; \hat{\beta} \;=\; \underset{\beta}{\min} \; \sum_{i=1}^{N} \hat{\varepsilon}_i(\beta)^2 \;=\; \underset{\beta}{\min} \; \sum_{i=1}^{N} \big( y_i - \beta x_i \big)^2. \;}
$$

In words: among all possible values of $\beta$, $\hat\beta$ is the one for which the sum of squared residuals $\sum_i \hat\varepsilon_i(\beta)^2$ is as small as possible. We square the residuals because positive and negative errors should not cancel out and large errors are penalized more than small ones. Squaring makes us indifferent between over- and under-estimation, while penalizing larger deviations more heavily; summing the squared residuals across all $N$ observations then gives the total error of the fit.

The OLS method - *Ordinary Least Squares* - is exactly the solution $\hat\beta$ of this program: the value of $\beta$ that minimizes the **sum of squared residuals**, producing the line that best fits the data in the least-squares sense. The term *“ordinary”* simply refers to this being the most basic and widely used version of least-squares estimation. Once we insert $\hat\beta$ back into $\hat{\varepsilon}_i(\beta)$, we recover the usual (estimated) residuals $\hat{\varepsilon}_i = \hat{\varepsilon}_i(\hat\beta) = y_i - \hat\beta x_i$.



**Seeing the least-squares program in action**

Before solving the program analytically (next subsection), let's *see* what it means to try out different candidate values of $\beta$. The animation below cycles through a handful of candidate lines (not every possible value of $\beta$ - plotting all of them at once on the same graph would be unreadable). For each candidate line it draws, in grey, the vertical segments joining each data point to the line: these are the residuals $\hat\varepsilon_i(\beta)$ for that candidate $\beta$. The title reports the resulting sum of squared residuals (SSR). Watch how the SSR first decreases, reaches a minimum, then increases again as $\beta$ moves away from the OLS estimate.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# A handful of candidate slopes to try. Showing every possible line at once would clutter
# the plot, so instead we animate through just a few - some too flat, some too steep, one
# close to the OLS solution.
candidate_betas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2]

fig, ax = plt.subplots(figsize=(8, 5))
x_line = np.linspace(x.min(), x.max(), 100)  # grid of x values used to draw each candidate line

def draw_frame(beta_candidate):
    ax.clear()

    # The data
    ax.scatter(x, y, alpha=0.4, color='red', label='data')

    # The candidate line y = beta_candidate * x
    ax.plot(x_line, beta_candidate * x_line, color='blue',
            label=fr'candidate line: $\hat y={beta_candidate:.2f}\,x$')

    # Residuals for this candidate beta: vertical segments from each point to the line
    y_pred = beta_candidate * x
    for xi, yi, ypi in zip(x, y, y_pred):
        ax.plot([xi, xi], [yi, ypi], color='grey', alpha=0.3, linewidth=1)

    ssr = np.sum((y - y_pred) ** 2)  # sum of squared residuals for this candidate beta

    ax.set_xlabel('Independent variable (x)', size=14)
    ax.set_ylabel('Dependent variable (y)', size=14)
    ax.set_title(fr'Candidate $\beta={beta_candidate:.2f}$   -   sum of squared residuals = {ssr:.2f}', size=14)
    ax.legend(loc='upper left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def update(frame):
    draw_frame(candidate_betas[frame])

anim = FuncAnimation(fig, update, frames=len(candidate_betas), interval=1200, repeat=True)
plt.close(fig)  # prevents a static duplicate of the last frame from being displayed below

HTML(anim.to_jshtml())


You can try changing the coefficients of the candidate line yourself directly in the code. Can you estimate the slope of the line that minimizes the sum of the squares?

### 4.2 Solving the minimization problem
To find $\beta$ that minimizes the error, we take the derivative of the sum with respect to $\beta$ and set it equal to zero (FOC):

$$
\frac{\partial}{\partial \beta} \sum_{i=1}^N \big( y_i - \beta x_i  \big)^2= 
-2\sum_{i=1}^N x_i \big( y_i - \beta x_i  \big) = 0
$$

Re-arranging, we get:

$$
\sum_{i=1}^N x_i y_i - \beta \sum_{i=1}^N x_i^2 = 0 \Rightarrow \quad \boxed{\hat{\beta} = \frac{\sum_{i=1}^N  x_i y_i}{\sum_{i=1}^N x_i^2}}
$$

This is the closed-form expression for the OLS estimator in the simple regression model through the origin.


### 4.3 Expressing $\hat{\beta}$ in terms of the error
We then substitute $y_i = \beta x_i + \varepsilon_i$ into the formula for $\hat{\beta}$:

$$
\hat{\beta}
= \frac{\sum_i x_i(\beta x_i + \varepsilon_i)}{\sum_i x_i^2}
= \frac{\beta\sum_i x_i^2+\sum_i x_i\varepsilon_i}{\sum_i x_i^2}
= \beta + \frac{\sum_i x_i \varepsilon_i }{\sum_i x_i^2}.
$$

Hence the estimation error is:

$$
\hat{\beta} - \beta = \frac{\sum_i x_i \varepsilon_i }{\sum_i x_i^2}.
$$

This expression will be essential to study the bias and the variance of $\hat{\beta}$.

### 4.4 Zero conditional mean

To study the expectation of $\hat{\beta}$ we impose a key assumption:
> $$\forall i \quad  \mathbb{E}[\varepsilon_i \mid x_i] = 0.$$

This is the **zero conditional mean assumption**. This assumption says that the expected value of the error term given $x_i$ is zero. Thus knowing $x_i$ tells us nothing systematic about the error term.  It means that, once we condition on $x_i$, the error term has no systematic tendency to be positive or negative. Intuitively: there is no systematic part of $y_i$ left in $\varepsilon_i$ that is correlated with $x_i$.

Using this assumption, we compute the expectation of the estimation error:
$$
\mathbb{E}[\hat{\beta}-\beta] = \mathbb{E}\left[\frac{\sum_i x_i \varepsilon_i }{\sum_i x_i^2}\right].
$$

The denominator $\sum_i x_i^2$ does not involve $\varepsilon_i$, so we can treat it as non-random (or condition on $x_i$). We focus on the numerator:

$$
\mathbb{E}\!\left[\sum_i x_i \varepsilon_i \right]
= \sum_i \mathbb{E}[x_i \varepsilon_i ]
= \sum_i \mathbb{E}\!\left[\mathbb{E}(x_i\varepsilon_i
\mid x_i)\right]
= \sum_i \mathbb{E}\!\left[x_i\,\mathbb{E}(\varepsilon_i \mid x_i)\right]
= 0.
$$

where we have $\mathbb{E}[x_i \varepsilon_i ] = \mathbb{E}\!\left[\mathbb{E}[x_i\varepsilon_i \mid x_i]\right]$ using the law of total expectation.

Therefore:
$$
\mathbb{E}[\hat{\beta}-\beta] = 0 \quad \Rightarrow \mathbb{E}[\hat{\beta}] = \beta.
$$

This means that $\hat{\beta}$ is an **unbiased estimator** of $\beta$. Putting everything together, we have shown:

$$
\boxed{
\hat{\beta} = \frac{\sum_i x_i y_i}{\sum_i x_i^2}
= \beta + \frac{\sum_i x_i\varepsilon_i}{\sum_i x_i^2} \quad \text{and} \quad \mathbb{E}[\hat{\beta}] = \beta \; \text{under} \; \mathbb{E}[\varepsilon_i \mid x_i] = 0.}
$$


**Remark** Zero conditional mean vs independence

Sometimes you may see a stronger assumption stating:
> $\forall i \quad x_i$ and $\varepsilon_i$ are independent.

This assumption implies in particular that $\mathbb{E}[x_i\varepsilon_i] = \mathbb{E}[x_i]\cdot \mathbb{E}[\varepsilon_i]$ and $\mathbb{E}[\varepsilon_i] = 0$.

However, for unbiasedness of $\hat{\beta}$ we do not need full independence.
The weaker and standard assumption in econometrics is:
> $$\forall i \quad  \mathbb{E}[\varepsilon_i \mid x_i] = 0.$$

Here's the insight: we've assumed that $\epsilon_i$ represents random noise. Since this noise is random, it cannot be correlated with $x$, that is $ x_i $ and $ \epsilon_i $ are independent and $ E(x_i \epsilon_i) = E(x_i) \cdot E(\epsilon_i) $. Given that $ E(\epsilon_i) = 0 $, we have $ E\left(\sum_j x_i \epsilon_i\right) = \sum_i 0 = 0 $. It means that the term $\sum_i x_i \epsilon_i$ will be zero in expectation (or close to zero when the number of observations is large).


We can verify this result using the graph below. On the graph, the scatter points represent the actual data points $(x_i, y_i)$ on the graph. The blue line is the best fit line that minimizes the total squared distance between the data points and the line, which is the line of predicted values. The residuals are the vertical distances between the scatter points and the regression line.


In [ ]:
numerator = np.sum(y*x)
denominator = np.sum(x*x)
bhat = numerator/denominator

yhat = x*bhat 

# Create Figure
fig, ax = plt.subplots(figsize=(10,5))

ax.scatter(x,y, alpha = 0.4, color = 'red')
ax.plot(x,yhat, color = 'blue')

ax.set_xlabel('Independent variable (x)')
ax.set_ylabel('Dependent variable (y)')
ax.set_title(r'Proposed DGP with $\beta$='+str(beta),size=20)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.annotate(fr'$\hat\beta=${bhat:.2f}', xy=(0.05, 0.9), xycoords='axes fraction', size=16, color='blue')

plt.show()

**Python code explanation**
- ``numerator = np.sum(y * x)``: This calculates the sum of the product of the observed values of Y and X, which is used in calculating the estimate for $\hat{\beta}$.
- ``denominator = np.sum(x * x)``: This calculates the sum of the squares of X, which is part of the denominator for calculating $\hat{\beta}$
- ``bhat = numerator / denominator``: This divides the sum of products by the sum of squares to get the estimated value of $\hat{\beta}$, which minimizes the least squares error.
- ``yhat = x * bhat``: This computes the predicted values of Y using the estimated $\hat{\beta}$.

The code uses Matplotlib to create a scatter plot of the data points (y vs x) in red with a transparency of 0.4 (alpha = 0.4). It then plots the estimated regression line (yhat vs x) in blue. Labels for the axes and title are set, and the top and right spines are removed for a cleaner plot.
Annotation:

The code annotates the plot with the estimated $\hat{\beta}$ value at coordinates (2.2, 3.5).

### Link with matrix notation

We propose here to connect the Python code to the matrix notation by explaining the key calculations in the mathematical framework of linear regression. The relationship between the dependent variable $ y $ and the independent variable $ x $ in matrix form is given by:
$$
y = x \beta + \epsilon
$$
where:
- $ y $ is the $ N \times 1 $ vector of observed dependent variables.
- $ x $ is the $ N \times p $ matrix of independent variables (in your example, $ x $ is a column vector since there is one independent variable).
- $ \beta $ is the $ p \times 1 $ vector of coefficients to be estimated.
- $ \epsilon $ is the $ N \times 1 $ vector of residuals (errors).

The ordinary least squares (OLS) estimate for $ \beta $ is:
$$
\hat{\beta} = \frac{x^\top y}{x^\top x}
$$
This formula minimizes the sum of squared errors.

### Mapping to the Python code

1. **Calculate $\hat{\beta}$:**
   - The numerator in the code, `np.sum(y * x)`, corresponds to $ x^\top y $, the inner product of $ x $ and $ y $.
   - The denominator, `np.sum(x * x)`, corresponds to $ x^\top x $, the inner product of $ x $ with itself.
   - The division `numerator / denominator` gives $ \hat{\beta} $, the estimated coefficient.

2. **Predict $ y $:**
   - `yhat = x * bhat` calculates the predicted values, corresponding to $ \hat{y} = x \hat{\beta} $.

3. **Plot the Data and Model:**
   - The scatter plot visualizes the data points ($ y $ vs. $ x $).
   - The line plot shows the fitted regression line ($ \hat{y} $ vs. $ x $).

4. **Annotation:**
   - The annotation in the plot highlights the estimated $ \hat{\beta} $ value, providing a numerical summary of the slope of the regression line.

### Python code and matrix notation side-by-side

$$
\begin{array}{|c|c|c|c|}
    \hline \textbf{Python code} & \textbf{Matrix notation} & \textbf{Python Object} & \textbf{Explanation} \\ 
    \hline
    \texttt{numerator = np.sum(y * x)} & x^\top y & \texttt{float} &  \text{Inner product of } x \text{ and } y \\
    \hline
    \texttt{denominator = np.sum(x * x)} & x^\top x & \texttt{float}  & \text{Inner product of } x \text{ with itself} \\
    \hline
    \texttt{bhat = numerator / denominator} & \hat{\beta} = \frac{x^\top y}{x^\top x} & \texttt{float}  & \text{Estimate the coefficient minimizing } \\
    &  & & \text{the least squares error}  \\
    \hline
    \texttt{yhat = x * bhat} & \hat{y} = x \hat{\beta} & \texttt{numpy.ndarray} \; \text{(size N)}  & \text{Predicted values using the estimated } \hat{\beta} \\
    \hline
\end{array}
$$

### Stata code

<details>
<summary>*Click here to see the Stata code*</summary>

```Stata
* Set seed for reproducibility
set seed 10

* Define parameters
local beta = 1
local N = 100

* Generate data
gen x = exp(runiform())
gen eps = rnormal(0, 1)
gen y = `beta' * x + eps

* Estimate beta
gen bhat = sum(y * x) / sum(x * x)
gen yhat = x * bhat

* Plotting
scatter y x, mcolor(red) msymbol(o) msize(0.5) xlabel(, grid) ylabel(, grid)
line yhat x, lcolor(blue) 
text 2.2 3.5 "β̂ = " + string(bhat, "%9.2f"), color(blue) size(medium)
```

**Stata code explanation**

- ``gen x = exp(runiform()``: Generates N random values from an exponential distribution for x.
- ``gen eps = rnormal(0, 1)``: Generates N random errors (residuals) from a normal distribution.
- ``gen y = \beta'* x + eps``: Calculates y as the dependent variable in the model $y = \beta x + \epsilon$
- ``gen bhat = sum(y * x) / sum(x * x)``: Computes the estimated $\hat{\beta}$.
- ``gen Yhat = x * bhat``: Calculates the predicted y values using the estimated $\hat{\beta}$.
- ``scatter y x``: Creates a scatter plot with red dots.
- ``line yhat x``: Plots the regression line.
- ``text``...: Annotates the plot with the estimated $\hat{\beta}$.

</details>
    

### R code

<details>
<summary>*Click here to see the Stata code*</summary>

    
```R
# Set seed for reproducibility
set.seed(10)

# Define parameters
beta <- 1
N <- 100

# Generate data
x <- rexp(N, rate = 1)
eps <- rnorm(N, mean = 0, sd = 1)
y <- beta * x + eps

# Estimate beta
numerator <- sum(y * x)
denominator <- sum(x * x)
bhat <- numerator / denominator
yhat <- x * bhat

# Plotting
plot(x, y, pch = 16, col = rgb(1, 0, 0, alpha = 0.4), xlab = "Independent Variable (x)", 
     ylab = "Dependent Variable (y)", main = paste("Proposed DGP with β =", beta))
lines(x, yhat, col = "blue")
text(2.2, 3.5, paste("β̂ =", round(bhat, 2)), col = "blue", cex = 1.5)

```

**R code explanation**

- ``x <- rexp(N, rate = 1)``: Generates N random values from an exponential distribution for the independent variable x.
- ``eps <- rnorm(N, mean = 0, sd = 1)``: Generates N random errors (residuals) from a normal distribution.
- ``y <- beta * x + eps``: Calculates the dependent variable Y based on the model $y = \beta x + \epsilon$.
Estimate Beta ($\hat{\beta}$):

- ``numerator <- sum(y * x)``: Calculates the sum of the product of the observed values of Y and X.
- ``denominator <- sum(x * x)``: Calculates the sum of the squares of x.
- ``bhat <- numerator / denominator``: Calculates the estimated $\hat{\beta}$.
- ``plot(x, y, ...)``: Creates a scatter plot with transparency (alpha = 0.4) for better visualization.
- ``lines(x, yhat, col = "blue")``: Plots the regression line.
- ``text(2.2, 3.5, ...)``: Adds the estimated value of $\hat{\beta}$ on the plot.

</details>

### <font color='orange'>Task #2:</font> Standardize $x$
We now standardize the values of $x_i$ before generating $y_i$. Instead of using the original $x_i$, we work with its standardized version:
$$
x_i^{\text{std}} = \frac{x_i - \bar{x}}{\text{sd}(x)},
$$
where $\bar{x}$ is the sample mean of $x$, and $\text{sd}(x)$ is its sample standard deviation.

In the code, this transformation is implemented as:
```Python
x_standardized = (x - np.mean(x)) / np.std(x)
```

Replace ``x`` by ``x_standardized`` in the code that generates $y$, re-run the code, and compare the scatter plot and the estimated value of $\hat{\beta}$ and its interpretation, to the case where you used the original (non-standardized) $x$.

Answer the following questions:

1. How does standardizing $x$ affect the interpretation of $\beta$?

Hint: before standardization, $\beta$ measures the effect of a one-unit change in $x$; after standardization, what does a “one-unit” change in $x^{std} correspond to?)

2. Does the distribution of $y$ change when you standardize $x$? Why or why not?

### <font color='teal'>Quick check #1</font>

**1. What is the "least-squares program" formally solving for?**

a) The value of $y$ that best fits $x$.<br>
b) The value of $\beta$ that minimizes the sum of squared residuals $\sum_i \hat\varepsilon_i(\beta)^2$.<br>
c) The value of $\varepsilon$ that maximizes the fit.<br>
d) The largest possible value of $x$.

<details>
<summary>Show answer</summary>

Answer: **b)**. $\hat\beta=\arg\min_\beta \sum_i \hat\varepsilon_i(\beta)^2$ - we search over candidate $\beta$'s for the one making the total squared residual smallest.

</details>

**2. Solving the first-order condition of the least-squares program gives the closed-form estimator:**

a) $\hat\beta = \dfrac{\sum_i y_i}{\sum_i x_i}$<br>
b) $\hat\beta = \dfrac{\sum_i x_i y_i}{\sum_i x_i^2}$<br>
c) $\hat\beta = \dfrac{\sum_i x_i}{N}$<br>
d) $\hat\beta = \dfrac{\sum_i y_i^2}{\sum_i x_i^2}$

<details>
<summary>Show answer</summary>

Answer: **b)**. Setting the derivative of $\sum_i(y_i-\beta x_i)^2$ with respect to $\beta$ to zero and rearranging gives exactly this closed-form solution.

</details>

**3. The zero conditional mean assumption $\mathbb{E}[\varepsilon_i\mid x_i]=0$ is used to show that:**

a) $\varepsilon_i$ is always exactly zero.<br>
b) $\hat\beta$ is unbiased, i.e. $\mathbb{E}[\hat\beta]=\beta$.<br>
c) $x_i$ and $y_i$ are uncorrelated.<br>
d) The variance of $\hat\beta$ is zero.

<details>
<summary>Show answer</summary>

Answer: **b)**. Under $\mathbb{E}[\varepsilon_i\mid x_i]=0$, the estimation error $\hat\beta-\beta$ has expectation zero, so $\hat\beta$ is an unbiased estimator of $\beta$ - it does **not** mean $\varepsilon_i$ is zero in any given sample.

</details>

---


## 5. Variance of our estimate

Our estimate of $\beta$ is accurate, but it's not exactly the true value. Why? So far, we have seen how OLS provides an estimator $\hat{\beta}$ of the true parameter $\beta$.  
Even if $\hat{\beta}$ is **unbiased** (i.e. $\mathbb{E}[\hat{\beta}] = \beta$), it will almost never be exactly equal to $\beta$ in a given sample, because the data contain randomness through the error terms $\varepsilon_i$.

To quantify how much $\hat{\beta}$ can fluctuate from sample to sample, we study its **variance**.


### 5.1 Variance of $\hat{\beta}$

Recall the simple linear regression model with one regressor $y_i = \beta x_i + \varepsilon_i$, $i = 1,\dots,N,$ and the OLS estimator:
$$
\hat{\beta} = \frac{\sum_{i=1}^N x_i y_i}{\sum_{i=1}^N x_i^2}.
$$

Using the population model $y_i = \beta x_i + \varepsilon_i$, we derived:
$$
\hat{\beta}
= \beta + \frac{\sum_{i=1}^N x_i \varepsilon_i}{\sum_{i=1}^N x_i^2}.
$$

This expression is very useful because it shows that the estimation error $\hat{\beta} - \beta$ comes entirely from the term $\sum_i x_i \varepsilon_i$ which depends on the random errors.

Assume:
- $\mathbb{E}[\varepsilon_i] = 0$,
- $\text{Var}(\varepsilon_i) = \sigma^2$,
- $\varepsilon_i$ are independent of each other,
- and $x_i$ are treated as non-random (or fixed) in repeated samples.

We can now calculate the variance of $\hat{\beta}$. We recall that:

$$
\text{Var}(\hat{\beta}) = \frac{1}{N} \sum_i \left[ (\hat{\beta} - \beta) x_i \right]^2
$$

It measures the spread of the estimated values of $\hat{\beta}$ around the true value, reflecting the uncertainty in our estimate. Then, since the $\varepsilon_i$ are independent and each has variance $\sigma^2$, we have:

$$
\text{Var}(\hat{\beta}) = \frac{1}{N} \sum_i \left[ \frac{\sum_k x_k \epsilon_k}{\sum_k x_k^2}. x_i \right]^2 = \frac{1}{N} \left[ \frac{\sum_k x_k \epsilon_k}{\sum_k x_k^2} \right]^2 \sum_i x_i^2 = \frac{1}{N}  \frac{\left[\sum_k x_k \epsilon_k\right]^2}{\sum_k x_k^2}= \frac{\sigma^2}{\sum_i x_i^2}.
$$

So we obtain the key formula:
$$
\boxed{
\text{Var}(\hat{\beta}) = \frac{\sigma^2}{\sum_i x_i^2}.
}
$$



We can interpret this formula as follows:

- $\sigma^2$ measures the **noise** in the model (how much the errors $\varepsilon_i$ vary around the regression line).
- $\sum_i x_i^2$ measures the **dispersion** of the regressor $x$ (how spread out the $x_i$ are).

If $x_i$ takes many different values far from zero (large $\sum_i x_i^2$), then $\hat{\beta}$ is more precise (lower variance). If the errors are very noisy (large $\sigma^2$), $\hat{\beta}$ is less precise (higher variance).

### 5.2 Estimating $\sigma^2$ in practice

In the formula:
$$
\text{Var}(\hat{\beta}) = \frac{\sigma^2}{\sum_i x_i^2},
$$
$\sigma^2 = \text{Var}(\varepsilon_i)$ is a **population** quantity: it is unknown.

In practice, we do not observe the true errors $\varepsilon_i$, only the **residuals**:
$$
\hat{\varepsilon}_i = y_i - \hat{y}_i
= y_i - \hat{\beta} x_i.
$$

A naive idea would be to estimate $\sigma^2$ by the average squared residual:
$$
\frac{1}{N} \sum_i \hat{\varepsilon}_i^2.
$$
However, this systematically **underestimates** the true variance because the same data have been used to estimate the parameter $\hat{\beta}$. Intuitively, the regression line is chosen to be “close” to the data, so the residuals are smaller on average than the true errors.

To correct for this, we divide not by $N$, but by the **degrees of freedom**:
$$
N - p,
$$
where $p$ is the number of parameters estimated in the regression (in simple regression with no intercept, we have $p=1$; with an intercept and one slope, we have $p=2$, etc.).

The usual **unbiased estimator** of $\sigma^2$ is therefore:
$$
\boxed{
\hat{\sigma}^2 = \frac{1}{N - p} \sum_{i=1}^N \hat{\varepsilon}_i^2.
}
$$

We then plug this estimate into the variance formula:
$$
\widehat{\text{Var}}(\hat{\beta})
= \frac{\hat{\sigma}^2}{\sum_i x_i^2}.
$$

The **standard error** of $\hat{\beta}$ is the square root of this estimated variance:
$$
\text{SE}(\hat{\beta}) = \sqrt{\widehat{\text{Var}}(\hat{\beta})}
= \sqrt{\frac{\hat{\sigma}^2}{\sum_i x_i^2}}.
$$

This standard error is what software (`statsmodels`) reports, and it will be the basis for constructing confidence intervals and t-tests in the next section.


### <font color='teal'>Quick check #2</font>

**1. The formula $\text{Var}(\hat\beta) = \dfrac{\sigma^2}{\sum_i x_i^2}$ tells us that $\hat\beta$ is more precise (lower variance) when:**

a) $\sigma^2$ is large and $\sum_i x_i^2$ is small.<br>
b) $\sigma^2$ is small and/or the $x_i$ are spread far from zero (large $\sum_i x_i^2$).<br>
c) $N$ is small.<br>
d) $\beta$ is large.

<details>
<summary>Show answer</summary>

Answer: **b)**. Less noise ($\sigma^2$ small) or more dispersion in the regressor ($\sum_i x_i^2$ large) both make the estimate more precise.

</details>

**2. Why do we divide by $N-p$ (degrees of freedom) rather than by $N$ when estimating $\sigma^2$?**

a) It is an arbitrary convention with no real justification.<br>
b) Dividing by $N$ would systematically underestimate $\sigma^2$, because the same data were already used to estimate $\hat\beta$.<br>
c) Because $p$ is always equal to $1$.<br>
d) To force $\hat\sigma^2$ to be negative.

<details>
<summary>Show answer</summary>

Answer: **b)**. The residuals are, on average, smaller than the true errors because the regression line was chosen to fit the data. Dividing by $N-p$ corrects this and gives an unbiased estimator of $\sigma^2$.

</details>

**3. The standard error $\text{SE}(\hat\beta)$ is:**

a) The variance of the residuals.<br>
b) The square root of the estimated variance of $\hat\beta$.<br>
c) Always equal to $1$.<br>
d) The same quantity as $\hat\beta$ itself.

<details>
<summary>Show answer</summary>

Answer: **b)**. $\text{SE}(\hat\beta)=\sqrt{\widehat{\text{Var}}(\hat\beta)}$ - it measures the typical size of the estimation error, in the same units as $\hat\beta$.

</details>

---


# 6. Standard error and t-test ([top](#home))<a id="ols"></a>
Now we want to understand relationships in practice. When we estimate a regression coefficient $\hat{\beta}$, we want to know not only its value but also how reliable it is. So far, we know how to estimate a coefficient $\hat{\beta}$ and its variance. From the previous section, we know that the **standard error** of $\hat{\beta}$ is:
$$
\text{SE}(\hat{\beta}) = \sqrt{\widehat{\text{Var}}(\hat{\beta})}.
$$

You can think of $\text{SE}(\hat{\beta})$ as a **typical error** we make when estimating $\beta$: if we repeated the same study many times with new samples, $\hat{\beta}$ would vary from sample to sample and the standard error measures how much.

### 6.1 Why do we use a t-test?

In practice, we rarely stop at the value of $\hat{\beta}$. We want to answer questions like:

> “Is the effect of $x$ on $y$ really equal to some value $\beta_0$  or could the difference between $\hat{\beta}$ and $\beta_0$ be just random noise?”

Formally, we choose a **null hypothesis** ($H_0)$ and an **alternative** ($H_1$). For a single coefficient, we usually test:
$$
H_0 : \beta = \beta_0\quad \text{and} \quad H_1 : \beta \neq \beta_0.
$$

A very common special case is $\beta_0 = 0$ (“no effect”) but the theory is identical for any $\beta_0$.

#### Which statistic to choose?

We know two facts from the previous sections: (1) $\hat{\beta}$ is a random variable centered around the true $\beta$, and (2) Its dispersion is summarized by the **standard error**:
   $$
   \text{SE}(\hat{\beta}) = \sqrt{\widehat{\text{Var}}(\hat{\beta})}.
   $$

If the null hypothesis $H_0: \beta = \beta_0$ is true, then $\hat{\beta}$ should be **close** to $\beta_0$ up to random sampling variation. A natural way to measure “how far” $\hat{\beta}$ is from $\beta_0$ is to take the difference $\hat{\beta} - \beta_0$ and scale it by its typical size, the standard error. This gives the **t-statistic**:
$$
t = \frac{\hat{\beta} - \beta_0}{\text{SE}(\hat{\beta})}.
$$

So the t-statistic answers:

> “How many standard errors away is $\hat{\beta}$ from the hypothesized value $\beta_0$?”

If $t$ is small in absolute value, $\hat{\beta}$ is close to $\beta_0$ relative to its uncertainty.  
If $|t|$ is large, the difference is big relative to the noise we expect.

To translate the difference “large” or “small” that is too imprecise into **probability**, we need to know the **distribution** of $t$ when $H_0$ is true.

#### Why t-distribution?

Under the usual OLS assumptions - errors normally distributed, variances estimated from the sample (using residuals), and use of all $N$ observations but also estimate $p$ parameters, one can show that if $H_0$ is true:
$$
t = \frac{\hat{\beta} - \beta_0}{\text{SE}(\hat{\beta})} \sim T_{N - p},
$$
where $T_{N-p}$ is a **t-distribution** with $\text{df} = N - p$ degrees of freedom.

Intuitively, the t-distribution arises because we do **two things at once**:

1. We use the data to estimate $\hat{\beta}$.
2. We also use the data to estimate the variance of the errors (and thus $\text{SE}(\hat{\beta})$).

This extra uncertainty (estimating the variance instead of knowing it) is exactly what creates the heavier tails of the t-distribution compared to the standard normal.


### 6.2 T-distribution in practice

The **t-distribution** looks like the standard normal (the familiar bell curve) but has heavier tails. This reflects more probability of “extreme” values when the sample is finite and the variance is estimated. The shape of the t-distribution depends on the degrees of freedom $\text{df} = N - p$. As $N$ increases (so $N - p$ increases), the t-distribution gets closer to the standard normal.

We can keep in minde this useful rule of thumb in many regressions: If $|t| \gtrsim 2$, the coefficient is often statistically significant at the 5% level (for reasonably large $N$).

In practice, statistical software reports for each coefficient:
- $\hat{\beta}$ and its standard error $\text{SE}(\hat{\beta})$,
- the t-statistics,
- and the **p-values**, which answers:

> “If $H_0$ were true, what is the probability of observing a t-statistic at least as extreme as the one we obtained?”

But conceptually, everything starts from the same object: the variability of the estimator summarized by $\text{SE}(\hat{\beta})$.


### 6.3 T-test step by step

First we set the degree of freedom $\text{df} = N - p$ and we choose a significance level $\alpha$ (probability of a “false alarm”), often $\alpha = 0.05$. We now summarize the t-test procedure in four steps.

**1. Set the hypotheses** with the null hypothesis:
     $
     H_0: \beta = \beta_0
     $
   and the alternative hypothesis:
     $
     H_1: \beta \neq \beta_0.
     $
     
**2. Compute the t-statistic** where $t$ is “how many standard errors” the estimate $\hat{\beta}$ is away from the null value $\beta_0$:

   $$
   t = \frac{\hat{\beta} - \beta_0}{\text{SE}(\hat{\beta})}.
   $$

   

**3. Compare to a critical value** or use the p-value

We can either:
   - look up the **critical value** $t^*$ in a t-table (see below),
   - or compute the **p-value**, which is the probability of observing a $t$-statistic as extreme as the one we got if $H_0$ were true.

   The decision rule for a two-sided test is now:
   - If $|t| > t^*$ (or equivalently p-value < $\alpha$): **reject $H_0$** → evidence of an effect.
   - If $|t| \le t^*$ (p-value ≥ $\alpha$): **do not reject $H_0$** → the data are compatible with “no effect”.
   

**4. Make an economic interpretation**
    
> Statistical significance does **not** automatically mean economic importance. Once you know $\hat{\beta}$ is significantly different from 0, you still need to interpret its magnitude in the economic context.


### 6.4 A simple example

Suppose we estimate a regression and obtain $\hat{\beta} = 1.2$ and $\text{SE}(\hat{\beta}) = 0.3$. Assume that in this regression the sample size is $N$ and we estimate $p$ parameters, so the degree of freedom is $\text{df} = N - p = 30$. We choose a 5% significance level for a two-sided test: $\alpha = 0.05$.

We want to answer the question:

> “Is there evidence that $x$ really affects $y$ or could this effect be just noise?”

**1. Set the hypotheses**:
     $
     H_0: \beta = 0
     $
   and
     $
     H_1: \beta \neq 0.
     $
Here, $\beta_0 = 0$.


**2. Compute the t-statistic:**

$$
t = \frac{\hat{\beta} - \beta_0}{\text{SE}(\hat{\beta})}
  = \frac{1.2 - 0}{0.3}
  = 4.0.
$$

We interpret it: the estimate $\hat{\beta}$ is 4 standard errors away from the null value $\beta_0 = 0$.


**3. Compare to a critical value:**

We already fixed $\text{df} = 30$ and $\alpha = 0.05$. From the t-table below, the critical value is $t^* \approx 2.042$. We compare:
$$
|t| = 4.0 \quad \text{vs} \quad t^* = 2.042.
$$

Since $|t| = 4.0 > 2.042 = t^*$, the observed t-statistic is too extreme to be explained by random noise if $H_0$ were true. So we **reject $H_0$** at the 5% level and conclude there is strong evidence that $\beta \neq 0$, i.e. that $x$ has an effect on $y$.


### 6.5 The t-table

The table below shows critical values $t^*$ for different degrees of freedom and tail probabilities. For a two-sided test (as the one we did) at level $\alpha$, the relevant column is the one whose entry corresponds to a total tail probability $\alpha$ (i.e. $\alpha/2$ in each tail).

| **df**  | **0.10**      | **0.05**      | **0.02**      | **0.01**          | **0.001**         |  
|---------|---------------|---------------|---------------|-------------------|-------------------|  
| 1       | 6.314         | 12.706        | 31.821        | 63.657            | 636.619           |  
| 2       | 2.920         | 4.303         | 6.965         | 9.925             | 31.599            |  
| 3       | 2.353         | 3.182         | 4.541         | 5.841             | 12.924            |  
| 4       | 2.132         | 2.776         | 3.747         | 4.604             | 8.610             |  
| 5       | 2.015         | 2.571         | 3.365         | 4.032             | 6.869             |  
| 10      | 1.812         | 2.228         | 2.764         | 3.169             | 4.587             |  
| 20      | 1.725         | 2.086         | 2.528         | 2.845             | 3.850             |  
| 30      | 1.697         | 2.042         | 2.457         | 2.750             | 3.646             |  
| 40      | 1.684         | 2.021         | 2.423         | 2.704             | 3.551             |  
| ∞       | 1.645         | 1.960         | 2.326         | 2.576             | 3.291             |  

How to use this table:

- Find the row for your degrees of freedom ($\text{df} = N - p$).
- Choose the column for your significance level $\alpha$ (e.g. 0.05).
- Read off the critical value $t^*$.
- Reject $H_0$ if $|t| > t^*$; otherwise, do not reject $H_0$.

For example:
- If $\text{df} = 10$, the critical value $t^* = 2.228$.
- If $\text{df} = 30$, the critical value $t^* = 2.042$.
- As $\text{df} \to \infty$, the t-distribution converges to the standard normal distribution with $t^* = 1.960$.

By applying the t-test, we determine whether $\hat{\beta}$ provides evidence of a significant relationship between $X$ and $Y$.


### 6.6 The limits of t-test

In our regression, $N$ denotes the number of observations (rows in the data), and $p$ the number of parameters estimated (columns in $\mathbf{X}$, including the constant if there is one). The degrees of freedom are then $\text{df} = N - p$.

This explains why both $N$ and $p$ matter for inference: if $p$ is large relative to $N$, we “use up” many degrees of freedom and our estimates become more uncertain. In practice, t-tables only report a limited set of df values. If your exact df is not in the table, you can either (i) take the closest value, or (ii) if df is sufficiently large, use the last row (often labelled $\infty$) which corresponds to the standard normal approximation.

More generally, choosing the appropriate test always depends on the **sample size** and the **assumptions** you make (normality of errors, homoskedasticity, independence, etc.). A careful econometrician first looks at the data (size, structure, number of regressors), checks which assumptions are plausible, and only then decides whether the usual t-tests are appropriate or if alternative methods (robust standard errors, non-parametric tests, other models) are needed.


### 6.7 Why the t-test (and not other tests)?

In this tutorial, we chose to focus on the t-test applied to regression coefficients. It is not the only statistical test used in econometrics, but it is the natural starting point for several reasons:

**1. Direct link with linear regression:** In OLS regression, the central question is:

   > “Does each explanatory variable have an effect different from zero on the dependent variable?”

   The t-test is designed to answer such questions about a mean or a coefficient estimated from a (approximately) Gaussian sample. It is therefore the basic tool to interpret each line of the regression table. For each coefficient $\beta_i$, we formalize this question with a test: $H_0 : \beta_i = \beta_{0i} \; \text{vs.} \; H_1 : \beta_i \neq \beta_{0i}$.

**2. It relies directly on the standard error:** The t-statistic measures “how many standard errors” the estimator $\hat{\beta}_i$ is away from the value under the null hypothesis $\beta_{0i}$. All inference based on the t-test therefore builds on the variance of $\hat{\beta}_i$. This makes the link very clear between the estimate $\hat{\beta}_i$, the uncertainty $\text{SE}(\hat{\beta}_i)$ and the statistical decision via the t-statistic.

**3. It generalizes easily:** In multiple regression, the (asymptotic) Gaussian approximation of OLS estimators allows us to keep using t-tests. This is exactly what econometrics software (Stata, R, Python, etc.) does for the `t` and `P>|t|` columns in the table.

#### How does t-test differ from other classical tests?

The concept of “test” is more general than that of t-test. Different tests can be used to answer different types of questions about data and are suited to different situations.

The t-test answers an individual question: is this particular coefficient significantly different from 
$\beta_0$? In regression, it is used on each coefficient to decide whether a given explanatory variable has a statistically significant effect on the dependent variable.

The F-test, in contrast, deals with joint questions. It does not look at one coefficient in isolation but asks whether a group of coefficients satisfy a set of restrictions. In a regression context, a typical null hypothesis is $H_0:\beta_2=\beta_3=0$: do two (or more) variables have no effect jointly? The t-test corresponds to a single equation whereas the F-test considers a system of equations. From this point of view, ANOVA (analysis of variance) is simply a particular use of the F-test: it compares means across several groups (for instance, average wages by education level) and is very close to a regression model where groups are represented by dummy variables.

Other tests answer questions that are further away from our basic OLS setting. The $\chi^2$ (chi-square) test is mainly used for categorical data and contingency tables: it compares observed and expected counts or tests independence between two discrete variables. The Kolmogorov–Smirnov test is a goodness-of-fit test: it compares an empirical distribution to a theoretical distribution (or two empirical distributions to each other). These tools are useful for checking distributional assumptions or for analysing discrete data but they do not intervene directly in the usual interpretation of linear regression coefficients.


#### Choosing the right test: role of $N$, $p$ and $\text{df}$

In our linear regression framework, the choice and interpretation of tests always depend on three basic ingredients: the number of observations $N$, the number of estimated parameters $p$, and the resulting degrees of freedom $\text{df}=N-p$. These quantities determine the reference distribution used for the test statistic (t or F), the precision of the estimators and the reliability of the conclusions.

A careful econometrician always starts from the structure of the problem: How large is the sample? How many parameters are we estimating? Are we interested in one coefficient or in a set of coefficients? Once the situation is identified, the corresponding tool follows quite naturally: t-tests for individual coefficients, F-tests for joint restrictions and possibly other tests for distributional assumptions or discrete outcomes.

In this tutorial, we focus on the t-test because these are the central tools to **read and interpret an OLS regression table**, which is the main objective for economics students here.

In [ ]:
resid = y-x*bhat
s =  np.sum(resid**2)/(N-1)
var = s/(np.sum(x**2))

se = np.sqrt(var)

# Create Figure
fig, ax = plt.subplots(figsize=(10,5))

ax.scatter(x,y, alpha = 0.4, color = 'red')
ax.plot(x,yhat, color = 'blue')

ax.set_xlabel('Independent variable (x)')
ax.set_ylabel('Dependent variable (y)')
ax.set_title(r'Proposed DGP with $\beta$='+str(beta),size=20)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.annotate(fr'$\hat\beta=${bhat:.2f} ({se:.4f})', xy=(0.05, 0.9), xycoords='axes fraction', size=16, color='blue')

plt.show()

From the graph, we can see that our simple linear model replicates the data well and - since we know the true DGP - also recovers the true value of $\beta$. 

In our simple example we only included one variable and therefore only solved for one coefficient. Going forward, our data will be much more complicated (e.g., more independent x variables) and we'll want to account for the possibility that our dependent variable attains some value independent of the observed X variables in our data - we'll want to include a constant in our specification of $f$. It turns out that doing all of this is pretty straight-forward and the intution gained from our simple example extends easilly. 

### <font color='teal'>Quick check #3</font>

**1. The t-statistic $t=\dfrac{\hat\beta-\beta_0}{\text{SE}(\hat\beta)}$ measures:**

a) The correlation between $x$ and $y$.<br>
b) How many standard errors $\hat\beta$ is away from the hypothesized value $\beta_0$.<br>
c) The p-value directly.<br>
d) The number of observations $N$.

<details>
<summary>Show answer</summary>

Answer: **b)**. The t-statistic rescales the gap $\hat\beta-\beta_0$ by its typical size $\text{SE}(\hat\beta)$, turning it into a number of standard errors.

</details>

**2. As the degrees of freedom $\text{df}=N-p$ grow large, the t-distribution:**

a) Becomes flatter, with heavier tails.<br>
b) Gets closer to the standard normal distribution.<br>
c) Turns into a chi-square distribution.<br>
d) Becomes undefined.

<details>
<summary>Show answer</summary>

Answer: **b)**. As $\text{df}\to\infty$, the extra uncertainty from estimating $\sigma^2$ becomes negligible, and the t-distribution converges to the standard normal.

</details>

**3. Suppose $\text{df}=30$, a two-sided test at $\alpha=0.05$ (critical value $t^\ast\approx 2.042$), and you compute $|t|=1.5$. What do you conclude?**

a) Reject $H_0$: $\beta$ has a statistically significant effect.<br>
b) Do not reject $H_0$ (since $1.5 < t^\ast$): the data are compatible with "no effect".<br>
c) The regression is invalid and must be re-estimated.<br>
d) $\hat\beta$ must be recomputed with more decimals.

<details>
<summary>Show answer</summary>

Answer: **b)**. Since $|t|=1.5$ is smaller than the critical value $t^\ast\approx 2.042$, the evidence is too weak to reject $H_0:\beta=\beta_0$ at the 5% level.

</details>

---


### <font color='orange'>Task #3</font>

1. Increase N to $\{200,500,1000\}$. What happens to the estimated coefficient $(\hat\beta)$ and the estimated standard errors $(\hat \epsilon)$?

2. Change the specification by changing $\beta$ to a value of your choice and re-run the code to find $\hat\beta$.